#  MobileNetV2 Feasibility Check
Mirrors `MRI_2_uncertainty_mcnemar.ipynb`, adapted for the MobileNetV2 checkpoint
(`mobilenetv2_clean.keras`, already trained under the corrected protocol in
Notebook 1, Cell 1). All outputs are saved with a `_mobilenet` suffix so
nothing overwrites the EfficientNetB3 results already used in the paper.

**Goal of this notebook:** answer one question — does the MC Dropout /
BatchNorm fix, rejection curve, and calibration approach transfer cleanly to
MobileNetV2, or does something break? Run top to bottom. If a cell errors out
in a way that needs real debugging (not just a path/name fix), stop and report
back — that's the signal to keep EfficientNetB3 (Strategy A) rather than switch
(Strategy B).


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, random, gc, warnings
import numpy as np
import tensorflow as tf
warnings.filterwarnings('ignore')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_ROOT = '/content/drive/MyDrive/MRI_Brain_Tumor_Project'
CLAHE_DIR    = f'{PROJECT_ROOT}/data/clahe_processed'
PRED_DIR     = f'{PROJECT_ROOT}/results/predictions'
METRICS_DIR  = f'{PROJECT_ROOT}/results/metrics'
FIGURES_DIR  = f'{PROJECT_ROOT}/results/figures'
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
CLASS_NAMES  = ['glioma', 'meningioma', 'notumor', 'pituitary']

# NOTE: MobileNetV2 needs its OWN preprocessing function -- this is the
# single most important line to get right. Using efficientnet.preprocess_input
# here by mistake would silently corrupt every downstream result.
preprocess_fn = tf.keras.applications.mobilenet_v2.preprocess_input

test_ds_check = tf.keras.utils.image_dataset_from_directory(
    f'{CLAHE_DIR}/Testing',
    seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='int', class_names=CLASS_NAMES, shuffle=False
).map(
    lambda x, y: (preprocess_fn(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

mobilenet_model = tf.keras.models.load_model(
    f'{PROJECT_ROOT}/models/checkpoints/mobilenetv2_clean.keras'
)

y_true          = np.load(f'{PRED_DIR}/y_true.npy')
y_pred_baseline = np.load(f'{PRED_DIR}/y_pred_baseline.npy')

print("Extracting test images (MobileNetV2 preprocessing)...")
X_list, y_list = [], []
for x_batch, y_batch in test_ds_check:
    X_list.append(x_batch.numpy())
    y_list.append(y_batch.numpy())
X_test  = np.concatenate(X_list, axis=0).astype(np.float32)
y_check = np.concatenate(y_list, axis=0)
del X_list, y_list
gc.collect()

print(f"GPU: {tf.config.list_physical_devices('GPU')}")
print(f"X_test: {X_test.shape}")
print(f"Labels match y_true: {np.array_equal(y_check, y_true)}")

y_pred_probs_mobilenet = mobilenet_model.predict(test_ds_check, verbose=0)
y_pred_mobilenet = np.argmax(y_pred_probs_mobilenet, axis=1)

acc_mobilenet = float(np.mean(y_pred_mobilenet == y_true))
print(f"\nMobileNetV2 test accuracy (recomputed here): {acc_mobilenet:.4f}")
print(f"Expected from Table V:                       0.9175")
print(f"Baseline (Scratch CNN) acc:                   {np.mean(y_pred_baseline == y_true):.4f}")

np.save(f'{PRED_DIR}/y_pred_mobilenet.npy', y_pred_mobilenet)
np.save(f'{PRED_DIR}/y_pred_probs_mobilenet.npy', y_pred_probs_mobilenet)
print("\nSaved y_pred_mobilenet.npy and y_pred_probs_mobilenet.npy")


## 1. Per-Class Report + Confusion Matrix (sanity check )

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json

print("="*55)
print("MobileNetV2 -- Per-Class Report (clean split)")
print("="*55)
report_mobilenet = classification_report(
    y_true, y_pred_mobilenet, target_names=CLASS_NAMES, digits=4, output_dict=True
)
print(classification_report(y_true, y_pred_mobilenet, target_names=CLASS_NAMES, digits=4))

weighted_f1_mobilenet = f1_score(y_true, y_pred_mobilenet, average='weighted')
print(f"Weighted F1 (MobileNetV2): {weighted_f1_mobilenet:.4f}")
print(f"Expected from Table V:     0.9157")

with open(f'{METRICS_DIR}/per_class_classification_report_mobilenet.json', 'w') as f:
    json.dump({'mobilenet': {'report': report_mobilenet,
                              'weighted_f1': float(weighted_f1_mobilenet)}}, f, indent=2)

cm = confusion_matrix(y_true, y_pred_mobilenet)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
plt.title('Confusion Matrix -- MobileNetV2 (CLAHE)')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/confusion_matrix_mobilenet.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved confusion_matrix_mobilenet.png")


## 2. McNemar's Test (Scratch CNN vs MobileNetV2)

In [ ]:
from scipy.stats import chi2
import numpy as np, json

baseline_correct  = (y_pred_baseline  == y_true)
mobilenet_correct = (y_pred_mobilenet == y_true)

both_correct  = np.sum( baseline_correct &  mobilenet_correct)
baseline_only = np.sum( baseline_correct & ~mobilenet_correct)
mobilenet_only= np.sum(~baseline_correct &  mobilenet_correct)
both_wrong    = np.sum(~baseline_correct & ~mobilenet_correct)

b = mobilenet_only
c = baseline_only

print("McNemar Contingency Table (Scratch CNN vs MobileNetV2)")
print(f"  Both correct:        {both_correct}")
print(f"  MobileNetV2 only:    {b}")
print(f"  Baseline only:       {c}")
print(f"  Both wrong:          {both_wrong}")

mcnemar_stat = (abs(b - c) - 1) ** 2 / (b + c)
p_value = 1 - chi2.cdf(mcnemar_stat, df=1)

print(f"\nChi-square statistic: {mcnemar_stat:.4f}")
print(f"p-value:              {p_value:.6f}")
sig = "STATISTICALLY SIGNIFICANT" if p_value < 0.05 else "NOT statistically significant"
print(f"Result: {sig} at alpha=0.05")

print(f"\nAccuracy summary:")
print(f"  Scratch CNN baseline: {np.mean(baseline_correct)*100:.2f}%")
print(f"  MobileNetV2:          {np.mean(mobilenet_correct)*100:.2f}%")

mcnemar_results_mobilenet = {
    'baseline_accuracy':  float(np.mean(baseline_correct)),
    'mobilenet_accuracy': float(np.mean(mobilenet_correct)),
    'both_correct': int(both_correct), 'mobilenet_only': int(b),
    'baseline_only': int(c), 'both_wrong': int(both_wrong),
    'mcnemar_statistic': float(mcnemar_stat), 'p_value': float(p_value),
    'significant': bool(p_value < 0.05), 'alpha': 0.05
}
with open(f'{METRICS_DIR}/mcnemar_results_mobilenet.json', 'w') as f:
    json.dump(mcnemar_results_mobilenet, f, indent=2)
print("\nSaved mcnemar_results_mobilenet.json")


## 3. MC Dropout -- the actual feasibility test




In [ ]:
import numpy as np, tensorflow as tf, gc
from collections import Counter

N_PASSES      = 50
N_CLASSES     = len(CLASS_NAMES)
BATCH_SIZE_MC = 32

print("Model layers (VERIFY this matches the expected shape before trusting the rest):")
for i, layer in enumerate(mobilenet_model.layers):
    print(f"  {i}: {layer.name} ({layer.__class__.__name__})")

backbone             = mobilenet_model.layers[1]
global_avg_pool      = mobilenet_model.layers[2]
dropout_layer        = mobilenet_model.layers[3]
classification_head  = mobilenet_model.layers[4]

assert isinstance(dropout_layer, tf.keras.layers.Dropout), (
    "STOP: layers[3] is not a Dropout layer for this architecture. "
    "The EfficientNetB3 layer-index assumption does not hold here -- "
    "inspect the printed layer list above and fix the indices manually "
    "before continuing. This is exactly the kind of break the feasibility "
    "check is designed to catch."
)

print(f"\nExtracting and pooling features (training=False, deterministic)...")
features_list = []
for start in range(0, len(X_test), BATCH_SIZE_MC):
    x_b = tf.constant(X_test[start:start+BATCH_SIZE_MC])
    back_feats   = backbone(x_b, training=False)
    pooled_feats = global_avg_pool(back_feats).numpy()
    features_list.append(pooled_feats)
features = np.concatenate(features_list, axis=0)
print(f"Features shape: {features.shape}")

det_check = np.argmax(
    classification_head(dropout_layer(tf.constant(features), training=False)).numpy(), axis=1
)
print(f"Deterministic acc from features: {np.mean(det_check == y_true):.4f}")
print(f"Matches y_pred_mobilenet:        {np.mean(det_check == y_pred_mobilenet):.4f}")
print("(If this second number isn't ~1.0, the layer split above is wrong -- stop and debug.)")

print(f"\nRunning {N_PASSES} MC Dropout passes...")
mc_mean = np.zeros((len(y_true), N_CLASSES), dtype=np.float64)
mc_m2   = np.zeros((len(y_true), N_CLASSES), dtype=np.float64)

for i in range(N_PASSES):
    batch_preds = []
    for start in range(0, len(features), BATCH_SIZE_MC):
        f_b  = tf.constant(features[start:start+BATCH_SIZE_MC])
        drop = dropout_layer(f_b, training=True)
        pred = classification_head(drop, training=False).numpy()
        batch_preds.append(pred)
    pass_preds = np.concatenate(batch_preds, axis=0)
    delta    = pass_preds - mc_mean
    mc_mean += delta / (i + 1)
    delta2   = pass_preds - mc_mean
    mc_m2   += delta * delta2
    del batch_preds, pass_preds
    if (i + 1) % 10 == 0:
        print(f"  Pass {i+1}/{N_PASSES} done")

mc_var        = np.maximum(mc_m2 / N_PASSES, 0)
mc_std        = np.sqrt(mc_var)
mc_pred_class_mobilenet = np.argmax(mc_mean, axis=1)
uncertainty_mobilenet   = mc_std.mean(axis=1)
confidence_mobilenet    = mc_mean.max(axis=1)

mc_accuracy = np.mean(mc_pred_class_mobilenet == y_true)
det_match   = np.mean(mc_pred_class_mobilenet == y_pred_mobilenet)

print(f"\nMC Dropout accuracy:            {mc_accuracy:.4f}")
print(f"Deterministic accuracy:         {np.mean(y_pred_mobilenet == y_true):.4f}")
print(f"MC matches deterministic preds: {det_match:.4f}")
print(f"Mean uncertainty:               {uncertainty_mobilenet.mean():.4f}")

if abs(mc_accuracy - np.mean(y_pred_mobilenet == y_true)) > 0.02:
    print("\n*** WARNING: MC accuracy diverges from deterministic accuracy by >2pp. ***")
    print("*** This is the same symptom as the original BatchNorm-Dropout bug. ***")
    print("*** Do not proceed to the rejection curve until this is understood. ***")
else:
    print("\nMC accuracy matches deterministic accuracy -- the fix transferred cleanly.")

np.save(f'{PRED_DIR}/mc_mean_mobilenet.npy',       mc_mean)
np.save(f'{PRED_DIR}/mc_std_mobilenet.npy',        mc_std)
np.save(f'{PRED_DIR}/mc_pred_class_mobilenet.npy', mc_pred_class_mobilenet)
np.save(f'{PRED_DIR}/uncertainty_mobilenet.npy',   uncertainty_mobilenet)
np.save(f'{PRED_DIR}/confidence_mobilenet.npy',    confidence_mobilenet)
print("\nSaved all MC Dropout arrays with _mobilenet suffix.")


## 4. Rejection Curve

In [ ]:
import numpy as np, matplotlib.pyplot as plt, json

sorted_idx = np.argsort(uncertainty_mobilenet)
rejection_fractions = np.arange(0, 0.91, 0.05)
retained_acc, retained_frac = [], []

for rej_frac in rejection_fractions:
    n_retain = int(len(y_true) * (1 - rej_frac))
    if n_retain == 0:
        break
    keep_idx = sorted_idx[:n_retain]
    acc = np.mean(mc_pred_class_mobilenet[keep_idx] == y_true[keep_idx])
    retained_acc.append(acc)
    retained_frac.append(1 - rej_frac)

retained_acc  = np.array(retained_acc)
retained_frac = np.array(retained_frac)

print("Rejection Curve Results (MobileNetV2):")
print(f"{'Retained %':>12} | {'Accuracy':>10}")
print("-"*28)
for rf, acc in zip(retained_frac, retained_acc):
    print(f"{rf*100:>11.0f}% | {acc*100:>9.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(retained_frac*100, retained_acc*100, 'o-', color='steelblue', linewidth=2, markersize=5)
axes[0].axhline(y=np.mean(mc_pred_class_mobilenet==y_true)*100, color='coral', linestyle='--',
                label=f'No rejection ({np.mean(mc_pred_class_mobilenet==y_true)*100:.1f}%)')
axes[0].set_xlabel('Retained samples (%)'); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Rejection Curve -- MobileNetV2 MC Dropout')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3); axes[0].set_xlim([10,100])

correct_mask  = mc_pred_class_mobilenet == y_true
unc_correct   = uncertainty_mobilenet[correct_mask]
unc_incorrect = uncertainty_mobilenet[~correct_mask]
axes[1].hist(unc_correct, bins=30, alpha=0.6, color='steelblue', label=f'Correct ({correct_mask.sum()})', density=True)
axes[1].hist(unc_incorrect, bins=30, alpha=0.6, color='coral', label=f'Incorrect ({(~correct_mask).sum()})', density=True)
axes[1].set_xlabel('Uncertainty (mean std)'); axes[1].set_ylabel('Density')
axes[1].set_title('Uncertainty Distribution: Correct vs Incorrect (MobileNetV2)')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/rejection_curve_mobilenet.png', dpi=150, bbox_inches='tight')
plt.show()

idx80 = np.argmin(np.abs(retained_frac - 0.8))
idx70 = np.argmin(np.abs(retained_frac - 0.7))
rejection_metrics_mobilenet = {
    'mc_accuracy_no_rejection': float(np.mean(mc_pred_class_mobilenet == y_true)),
    'acc_at_90pct_retained': float(retained_acc[np.argmin(np.abs(retained_frac - 0.9))]),
    'acc_at_80pct_retained': float(retained_acc[idx80]),
    'acc_at_70pct_retained': float(retained_acc[idx70]),
    'mean_uncertainty_correct': float(unc_correct.mean()),
    'mean_uncertainty_incorrect': float(unc_incorrect.mean()),
    'uncertainty_ratio': float(unc_incorrect.mean() / unc_correct.mean()),
}
with open(f'{METRICS_DIR}/rejection_curve_metrics_mobilenet.json', 'w') as f:
    json.dump(rejection_metrics_mobilenet, f, indent=2)
print("\nSaved rejection_curve_metrics_mobilenet.json")
print(json.dumps(rejection_metrics_mobilenet, indent=2))


## 5. Mann-Whitney U Tests (incorrect vs correct uncertainty -- the key significance check)

In [ ]:
from scipy import stats
import numpy as np, json

print("Mann-Whitney U Test -- Correct vs Incorrect Uncertainty (MobileNetV2)")
correct_mask = mc_pred_class_mobilenet == y_true
stat, p = stats.mannwhitneyu(
    uncertainty_mobilenet[~correct_mask],
    uncertainty_mobilenet[correct_mask],
    alternative='greater'
)
ratio = uncertainty_mobilenet[~correct_mask].mean() / uncertainty_mobilenet[correct_mask].mean()
print(f"  U statistic: {stat:.2f}")
print(f"  p-value:     {p:.6e}")
print(f"  Incorrect predictions carry {ratio:.2f}x higher uncertainty than correct ones")
print(f"  Significant (p<0.05): {p < 0.05}")

mannwhitney_mobilenet = {
    'statistic': float(stat), 'p_value': float(p),
    'uncertainty_ratio': float(ratio), 'significant': bool(p < 0.05)
}
with open(f'{METRICS_DIR}/mannwhitney_mobilenet.json', 'w') as f:
    json.dump(mannwhitney_mobilenet, f, indent=2)
print("\nSaved mannwhitney_mobilenet.json")


## 6. Calibration (ECE + Temperature Scaling)



In [ ]:
import numpy as np, matplotlib.pyplot as plt, tensorflow as tf, json

def compute_ece(probs, labels, n_bins=15):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    correct     = (predictions == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece, bin_data = 0.0, []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            bin_data.append({'conf': (lo+hi)/2, 'acc': 0, 'n': 0}); continue
        bin_acc, bin_conf, bin_n = correct[mask].mean(), confidences[mask].mean(), mask.sum()
        ece += (bin_n/len(labels)) * abs(bin_acc - bin_conf)
        bin_data.append({'conf': bin_conf, 'acc': bin_acc, 'n': int(bin_n)})
    return float(ece), bin_data

def apply_temperature(probs, T):
    eps = 1e-7
    logits = np.log(np.clip(probs, eps, 1-eps))
    scaled = logits / T
    e = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def find_optimal_temperature(probs, labels, T_range=None):
    if T_range is None:
        T_range = np.linspace(0.5, 3.0, 50)
    best_T, best_ece = 1.0, float('inf')
    for T in T_range:
        ece, _ = compute_ece(apply_temperature(probs, T), labels)
        if ece < best_ece:
            best_ece, best_T = ece, T
    return best_T, best_ece

# Rebuild the SAME validation split used to train MobileNetV2 (Notebook 1, Cell 1:
# validation_split=0.1, subset='validation', seed=42) -- fit temperature here, NOT on test.
val_ds_calib = tf.keras.utils.image_dataset_from_directory(
    f'{CLAHE_DIR}/Training', validation_split=0.1, subset='validation', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int',
    class_names=CLASS_NAMES, shuffle=False
).map(lambda x, y: (preprocess_fn(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

val_probs_list, val_labels_list = [], []
for x_b, y_b in val_ds_calib:
    val_probs_list.append(mobilenet_model(x_b, training=False).numpy())
    val_labels_list.append(y_b.numpy())
val_probs  = np.concatenate(val_probs_list, axis=0)
val_labels = np.concatenate(val_labels_list, axis=0)
print(f"Validation split for calibration: {val_probs.shape[0]} images")

ece_before, bins_before = compute_ece(y_pred_probs_mobilenet, y_true)
print(f"\nTest-set ECE before calibration: {ece_before:.4f}")

T_opt, _ = find_optimal_temperature(val_probs, val_labels)
print(f"Optimal temperature T (fit on VALIDATION split): {T_opt:.4f}")

probs_calibrated = apply_temperature(y_pred_probs_mobilenet, T_opt)
ece_after, bins_after = compute_ece(probs_calibrated, y_true)
print(f"Test-set ECE after calibration:  {ece_after:.4f}")
print(f"ECE improvement: {(ece_before-ece_after)/ece_before*100:.2f}%")

acc_before = np.mean(np.argmax(y_pred_probs_mobilenet, axis=1) == y_true)
acc_after  = np.mean(np.argmax(probs_calibrated, axis=1) == y_true)
print(f"\nAccuracy unchanged by temperature scaling: {abs(acc_before-acc_after) < 1e-6}")

ece_results_mobilenet = {
    'ece_before': ece_before, 'ece_after': ece_after,
    'optimal_temperature': T_opt,
    'fit_on': 'held-out validation split (NOT test set)',
    'improvement_pct': float((ece_before-ece_after)/ece_before*100)
}
with open(f'{METRICS_DIR}/calibration_ece_mobilenet.json', 'w') as f:
    json.dump(ece_results_mobilenet, f, indent=2)
print("\nSaved calibration_ece_mobilenet.json")


## 7. Multi-class ROC-AUC (One-vs-Rest)

In [ ]:
import numpy as np, matplotlib.pyplot as plt, json
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

y_true_bin = label_binarize(y_true, classes=[0,1,2,3])
fpr, tpr, roc_auc = {}, {}, {}
for i, cls_name in enumerate(CLASS_NAMES):
    fpr[cls_name], tpr[cls_name], _ = roc_curve(y_true_bin[:,i], y_pred_probs_mobilenet[:,i])
    roc_auc[cls_name] = auc(fpr[cls_name], tpr[cls_name])
macro_auc = roc_auc_score(y_true_bin, y_pred_probs_mobilenet, multi_class='ovr', average='macro')

print("Multi-class ROC-AUC (MobileNetV2)")
for cls_name in CLASS_NAMES:
    print(f"  {cls_name:>12}: AUC = {roc_auc[cls_name]:.4f}")
print(f"\n  Macro-average AUC: {macro_auc:.4f}")

fig, ax = plt.subplots(figsize=(9,7))
colors = ['steelblue','coral','green','purple']
for i, cls_name in enumerate(CLASS_NAMES):
    ax.plot(fpr[cls_name], tpr[cls_name], color=colors[i], linewidth=2,
            label=f'{cls_name.capitalize()} (AUC={roc_auc[cls_name]:.4f})')
ax.plot([0,1],[0,1],'k--',alpha=0.5,label='Random (AUC=0.50)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'Multi-class ROC -- MobileNetV2\nMacro-average AUC = {macro_auc:.4f}')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/roc_auc_curves_mobilenet.png', dpi=150, bbox_inches='tight')
plt.show()

roc_results_mobilenet = {cls: float(roc_auc[cls]) for cls in CLASS_NAMES}
roc_results_mobilenet['macro_average'] = float(macro_auc)
with open(f'{METRICS_DIR}/roc_auc_results_mobilenet.json', 'w') as f:
    json.dump(roc_results_mobilenet, f, indent=2)
print("\nSaved roc_auc_results_mobilenet.json")


## 8. Feasibility Verdict -- read this before deciding anything

In [ ]:
import json

print("="*60)
print("MOBILENETV2 FEASIBILITY CHECK -- SUMMARY")
print("="*60)

with open(f'{METRICS_DIR}/mcnemar_results_mobilenet.json') as f:
    mcn = json.load(f)
with open(f'{METRICS_DIR}/rejection_curve_metrics_mobilenet.json') as f:
    rej = json.load(f)
with open(f'{METRICS_DIR}/mannwhitney_mobilenet.json') as f:
    mw = json.load(f)
with open(f'{METRICS_DIR}/calibration_ece_mobilenet.json') as f:
    cal = json.load(f)

print(f"\nClassification accuracy:   {mcn['mobilenet_accuracy']*100:.2f}%  (expected 91.75%)")
print(f"vs Scratch CNN:            McNemar p={mcn['p_value']:.4f} "
      f"({'significant' if mcn['significant'] else 'not significant'})")
print(f"\nMC Dropout accuracy:       {rej['mc_accuracy_no_rejection']*100:.2f}%")
print(f"  -> matches deterministic acc within 2pp: "
      f"{abs(rej['mc_accuracy_no_rejection'] - mcn['mobilenet_accuracy']) < 0.02}")
print(f"Rejection @ 80% retained:  {rej['acc_at_80pct_retained']*100:.2f}%")
print(f"Uncertainty ratio (wrong/correct): {mw['uncertainty_ratio']:.2f}x "
      f"(p={mw['p_value']:.2e}, {'significant' if mw['significant'] else 'NOT significant'})")
print(f"\nCalibration: ECE {cal['ece_before']:.4f} -> {cal['ece_after']:.4f} "
      f"(T={cal['optimal_temperature']:.2f}, fit on validation split -- leakage-free)")



In [ ]:
import os, glob

PROJECT_ROOT = '/content/drive/MyDrive/MRI_Brain_Tumor_Project'

# 1. Kaunse checkpoints actually bane hain?
ckpt_dir = f'{PROJECT_ROOT}/models/checkpoints'
for f in sorted(os.listdir(ckpt_dir)):
    path = os.path.join(ckpt_dir, f)
    print(f, "-", os.path.getmtime(path))

# 2. Metrics folder mein kya kya hai — kahin 91.75/0.9157 wala koi aur file to nahi?
metrics_dir = f'{PROJECT_ROOT}/results/metrics'
for f in sorted(os.listdir(metrics_dir)):
    print(f)

# 3. Har metrics JSON ke andar "917" ya "9175" dhoondo (broad text search)
for f in glob.glob(f'{metrics_dir}/*.json'):
    content = open(f).read()
    if '917' in content or '9175' in content:
        print("FOUND in:", f)

In [ ]:
import json

# 1. Ye file "917" match kar rahi thi search mein — dekho asal mein kya hai
with open(f'{PROJECT_ROOT}/results/metrics/phase1_ece.json') as f:
    print("phase1_ece.json:", json.load(f))

# 2. Ye stale placeholder table ho sakti hai (notebook 2 cell 12 wali)
with open(f'{PROJECT_ROOT}/results/metrics/model_comparison.json') as f:
    print("\nmodel_comparison.json:", json.load(f))

# 3. Ye interesting hai — maine pehle iska zikr nahi kiya tha, ye already
#    ek validation-fit calibration attempt ho sakti hai
with open(f'{PROJECT_ROOT}/results/metrics/calibration_val_fit.json') as f:
    print("\ncalibration_val_fit.json:", json.load(f))

with open(f'{PROJECT_ROOT}/results/metrics/calibration_ece.json') as f:
    print("\ncalibration_ece.json:", json.load(f))

In [ ]:
import tensorflow as tf, numpy as np

def evaluate_checkpoint(name, ckpt_file, preprocess_fn, img_size):
    model = tf.keras.models.load_model(f'{PROJECT_ROOT}/models/checkpoints/{ckpt_file}')
    test_ds = tf.keras.utils.image_dataset_from_directory(
        f'{CLAHE_DIR}/Testing', seed=42, image_size=img_size, batch_size=32,
        label_mode='int', class_names=CLASS_NAMES, shuffle=False
    ).map(lambda x, y: (preprocess_fn(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    probs = model.predict(test_ds, verbose=0)
    preds = np.argmax(probs, axis=1)
    acc = float(np.mean(preds == y_true))
    print(f"{name}: accuracy = {acc:.4f}")
    return acc, probs

acc_resnet, _  = evaluate_checkpoint('ResNet50', 'resnet50_clean.keras',
                                       tf.keras.applications.resnet50.preprocess_input, (224,224))
acc_incep, _   = evaluate_checkpoint('InceptionV3', 'inceptionv3_clean.keras',
                                       tf.keras.applications.inception_v3.preprocess_input, (299,299))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np

PROJECT_ROOT = '/content/drive/MyDrive/MRI_Brain_Tumor_Project'
PRED_DIR = f'{PROJECT_ROOT}/results/predictions'

# Reload saved arrays -- NOT re-running MC Dropout, just calibration
y_true = np.load(f'{PRED_DIR}/y_true.npy')
y_pred_probs_mobilenet = np.load(f'{PRED_DIR}/y_pred_probs_mobilenet.npy')
print(f"Loaded: y_true {y_true.shape}, y_pred_probs_mobilenet {y_pred_probs_mobilenet.shape}")

def compute_ece(probs, labels, n_bins=15):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    correct = (predictions == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        bin_acc, bin_conf, bin_n = correct[mask].mean(), confidences[mask].mean(), mask.sum()
        ece += (bin_n/len(labels)) * abs(bin_acc - bin_conf)
    return float(ece)

def apply_temperature(probs, T):
    eps = 1e-7
    logits = np.log(np.clip(probs, eps, 1-eps))
    scaled = logits / T
    e = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def find_optimal_temperature(probs, labels, T_range=None):
    if T_range is None:
        T_range = np.linspace(0.5, 3.0, 50)
    best_T, best_ece = 1.0, float('inf')
    for T in T_range:
        ece = compute_ece(apply_temperature(probs, T), labels)
        if ece < best_ece:
            best_ece, best_T = ece, T
    return best_T, best_ece

# Split-based fix: fit T on one half of test set, report ECE on the other half
rng = np.random.RandomState(42)
idx = rng.permutation(len(y_true))
calib_idx, report_idx = idx[:480], idx[480:]

T_opt, _ = find_optimal_temperature(y_pred_probs_mobilenet[calib_idx], y_true[calib_idx])
ece_before = compute_ece(y_pred_probs_mobilenet[report_idx], y_true[report_idx])
probs_cal = apply_temperature(y_pred_probs_mobilenet, T_opt)
ece_after = compute_ece(probs_cal[report_idx], y_true[report_idx])

print(f"\nOptimal T (fit on 480-image split): {T_opt:.4f}")
print(f"ECE before: {ece_before:.4f}")
print(f"ECE after:  {ece_after:.4f}")
print(f"Improvement: {(ece_before-ece_after)/ece_before*100:.2f}%")